In [1]:
import os
import json
import random
import pickle
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 42

# Debug mode:
DEBUG = False

# Dataset sizes
if DEBUG:
    CONTROLLER_TRAIN_SIZE = 500
    CONTROLLER_VAL_SIZE = 100
    FINAL_TEST_SIZE = 100
else:
    CONTROLLER_TRAIN_SIZE = 5000
    CONTROLLER_VAL_SIZE = 1000
    FINAL_TEST_SIZE = 1000

DATASET_NAME = "hotpotqa/hotpot_qa"
DATASET_CONFIG = "distractor"

EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"

# Retrieval parameters
BM25_K = 5
DENSE_K = 10

# Embedding parameters
EMBED_BATCH_SIZE = 128

# Artifact directory
ARTIFACT_DIR = Path("/kaggle/working/retrieval_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("DEBUG:", DEBUG)
print("Controller train size:", CONTROLLER_TRAIN_SIZE)
print("Controller validation size:", CONTROLLER_VAL_SIZE)
print("Final test size:", FINAL_TEST_SIZE)
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Artifact directory:", ARTIFACT_DIR)

DEBUG: False
Controller train size: 5000
Controller validation size: 1000
Final test size: 1000
Embedding model: BAAI/bge-small-en-v1.5
Artifact directory: /kaggle/working/retrieval_artifacts


In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Deterministic behavior where practical.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch version:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu128
Device: cuda
GPU: Tesla T4


In [3]:
!pip install -q \
    datasets \
    sentence-transformers \
    rank-bm25 \
    faiss-cpu \
    pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.2 MB/s eta 0:00:00


In [4]:
import re
import time
import hashlib
from typing import Dict, List, Tuple, Any

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import faiss

from tqdm.auto import tqdm

In [5]:
print("Loading HotpotQA...")

dataset = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG
)

print(dataset)
print()
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))

Loading HotpotQA...


README.md: 0.00B [00:00, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 90447
    })
    validation: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 7405
    })
})

Train size: 90447
Validation size: 7405


In [6]:
example = dataset["train"][0]

print("ID:")
print(example["id"])

print("\nQuestion:")
print(example["question"])

print("\nAnswer:")
print(example["answer"])

print("\nType:")
print(example["type"])

print("\nDifficulty:")
print(example["level"])

print("\nContext titles:")
print(example["context"]["title"])

print("\nSupporting fact titles:")
print(example["supporting_facts"]["title"])

print("\nSupporting fact sentence IDs:")
print(example["supporting_facts"]["sent_id"])

ID:
5a7a06935542990198eaf050

Question:
Which magazine was started first Arthur's Magazine or First for Women?

Answer:
Arthur's Magazine

Type:
comparison

Difficulty:
medium

Context titles:
['Radio City (Indian radio station)', 'History of Albanian football', 'Echosmith', "Women's colleges in the Southern United States", 'First Arthur County Courthouse and Jail', "Arthur's Magazine", '2014–15 Ukrainian Hockey Championship', 'First for Women', 'Freeway Complex Fire', 'William Rast']

Supporting fact titles:
["Arthur's Magazine", 'First for Women']

Supporting fact sentence IDs:
[0, 0]


In [7]:
train_df = dataset["train"].to_pandas()
valid_df = dataset["validation"].to_pandas()

# Validate required columns.
required_columns = {
    "id",
    "question",
    "answer",
    "type",
    "level",
    "context",
    "supporting_facts",
}

missing_train = required_columns - set(train_df.columns)
missing_valid = required_columns - set(valid_df.columns)

if missing_train:
    raise ValueError(f"Missing train columns: {missing_train}")

if missing_valid:
    raise ValueError(f"Missing validation columns: {missing_valid}")


def stratified_sample(
    df: pd.DataFrame,
    n: int,
    seed: int,
    stratify_columns: List[str]
) -> pd.DataFrame:
    """
    Sample n rows while approximately preserving the distribution
    of the requested stratification columns.
    """
    if n > len(df):
        raise ValueError(
            f"Requested {n} rows, but only {len(df)} are available."
        )

    working = df.copy()

    # Combined stratification label.
    working["_stratum"] = (
        working[stratify_columns]
        .fillna("UNKNOWN")
        .astype(str)
        .agg("||".join, axis=1)
    )

    rng = np.random.default_rng(seed)

    groups = []
    total = len(working)

    for stratum, group in working.groupby("_stratum", sort=False):
        exact = len(group) / total * n
        base = int(np.floor(exact))
        remainder = exact - base

        groups.append({
            "stratum": stratum,
            "group": group,
            "base": base,
            "remainder": remainder,
        })

    allocated = sum(x["base"] for x in groups)
    remaining = n - allocated

    groups.sort(key=lambda x: x["remainder"], reverse=True)

    for i in range(remaining):
        groups[i]["base"] += 1

    sampled_parts = []

    for item in groups:
        group = item["group"]
        k = item["base"]

        if k > 0:
            sampled = group.sample(
                n=k,
                random_state=int(rng.integers(0, 2**31 - 1))
            )
            sampled_parts.append(sampled)

    result = pd.concat(sampled_parts, ignore_index=True)

    if len(result) != n:
        raise RuntimeError(
            f"Sampling error: expected {n}, got {len(result)}."
        )

    result = result.drop(columns=["_stratum"])

    return result


controller_train_df = stratified_sample(
    train_df,
    CONTROLLER_TRAIN_SIZE,
    SEED,
    ["type", "level"]
)

remaining_train_df = train_df[
    ~train_df["id"].isin(controller_train_df["id"])
].copy()

controller_val_df = stratified_sample(
    remaining_train_df,
    CONTROLLER_VAL_SIZE,
    SEED + 1,
    ["type", "level"]
)

final_test_df = stratified_sample(
    valid_df,
    FINAL_TEST_SIZE,
    SEED + 2,
    ["type", "level"]
)

print("Controller train:", len(controller_train_df))
print("Controller validation:", len(controller_val_df))
print("Final test:", len(final_test_df))

Controller train: 5000
Controller validation: 1000
Final test: 1000


In [8]:
train_ids = set(controller_train_df["id"])
val_ids = set(controller_val_df["id"])
test_ids = set(final_test_df["id"])

assert train_ids.isdisjoint(val_ids), "Train/validation overlap detected."
assert train_ids.isdisjoint(test_ids), "Train/test overlap detected."
assert val_ids.isdisjoint(test_ids), "Validation/test overlap detected."

print("No ID overlap between the three splits.")

print("\nTrain type distribution:")
print(controller_train_df["type"].value_counts(normalize=True))

print("\nValidation type distribution:")
print(controller_val_df["type"].value_counts(normalize=True))

print("\nFinal test type distribution:")
print(final_test_df["type"].value_counts(normalize=True))

No ID overlap between the three splits.

Train type distribution:
type
bridge        0.807
comparison    0.193
Name: proportion, dtype: float64

Validation type distribution:
type
bridge        0.807
comparison    0.193
Name: proportion, dtype: float64

Final test type distribution:
type
bridge        0.799
comparison    0.201
Name: proportion, dtype: float64


In [9]:
def normalize_example(row: pd.Series) -> Dict[str, Any]:
    context_titles = row["context"]["title"]
    context_sentences = row["context"]["sentences"]

    supporting_titles = row["supporting_facts"]["title"]
    supporting_sent_ids = row["supporting_facts"]["sent_id"]

    contexts = []

    for title, sentences in zip(context_titles, context_sentences):
        contexts.append({
            "title": str(title),
            "sentences": [str(s) for s in sentences],
        })

    return {
        "id": str(row["id"]),
        "question": str(row["question"]),
        "answer": str(row["answer"]),
        "type": str(row["type"]),
        "level": str(row["level"]),
        "contexts": contexts,
        "supporting_titles": [str(x) for x in supporting_titles],
        "supporting_sent_ids": [int(x) for x in supporting_sent_ids],
    }


controller_train_records = [
    normalize_example(row)
    for _, row in controller_train_df.iterrows()
]

controller_val_records = [
    normalize_example(row)
    for _, row in controller_val_df.iterrows()
]

final_test_records = [
    normalize_example(row)
    for _, row in final_test_df.iterrows()
]

print(controller_train_records[0].keys())
print(controller_train_records[0]["question"])

dict_keys(['id', 'question', 'answer', 'type', 'level', 'contexts', 'supporting_titles', 'supporting_sent_ids'])
Which city is larger, Pingxiang or Shijiazhuang?  


In [10]:
all_selected_records = (
    controller_train_records
    + controller_val_records
    + final_test_records
)

def make_doc_key(title: str, text: str) -> str:
    """
    Stable content-based key for deduplicating paragraphs.
    """
    normalized = (
        title.strip().lower()
        + "\n"
        + re.sub(r"\s+", " ", text.strip().lower())
    )

    return hashlib.sha1(
        normalized.encode("utf-8")
    ).hexdigest()


doc_by_key = {}
example_doc_keys = {}

for record in tqdm(
    all_selected_records,
    desc="Building paragraph corpus"
):
    example_id = record["id"]
    example_doc_keys[example_id] = []

    for paragraph_idx, (title, sentences) in enumerate(
        zip(
            [x["title"] for x in record["contexts"]],
            [x["sentences"] for x in record["contexts"]]
        )
    ):
        text = " ".join(sentences).strip()

        if not text:
            continue

        key = make_doc_key(title, text)

        if key not in doc_by_key:
            doc_by_key[key] = {
                "doc_key": key,
                "title": title,
                "text": text,
                "token_count_words": len(text.split()),
                "source_example_ids": [],
            }

        if example_id not in doc_by_key[key]["source_example_ids"]:
            doc_by_key[key]["source_example_ids"].append(example_id)

        example_doc_keys[example_id].append(key)


corpus_records = list(doc_by_key.values())

corpus_df = pd.DataFrame(corpus_records)

print("Unique retrieval documents:", len(corpus_df))
print(corpus_df.head())

Building paragraph corpus:   0%|          | 0/7000 [00:00<?, ?it/s]

Unique retrieval documents: 62933
                                    doc_key  \
0  5aa5382d9704e8ce00a14308877791f7a261aecd   
1  22d336eba40cc512e34e939b7f7b13e4bcabd710   
2  db7e1e57f757a29ce0d9d3ae58ee20442013441a   
3  46d8e4d572b34d543a530c95f9f0be727fcbb4dc   
4  d4524c0fe50de04c51580f7ae486e73ff2d75060   

                                          title  \
0                  Shijiazhuang Railway Station   
1  Shijiazhuang Zhengding International Airport   
2                Shijiazhuang Tiedao University   
3                                  Shijiazhuang   
4            Shijiazhuang North Railway Station   

                                                text  token_count_words  \
0  The Shijiazhuang Railway Station () is the mai...                 78   
1  Shijiazhuang Zhengding International Airport (...                 53   
2  Shijiazhuang Tiedao University (STDU), formerl...                 58   
3  Shijiazhuang ( ; ), formerly romanized Shihkia...                 41   
4

In [11]:
corpus_df = corpus_df.reset_index(drop=True)

corpus_df["doc_id"] = np.arange(
    len(corpus_df),
    dtype=np.int32
)

# Rebuild mapping.
doc_key_to_id = dict(
    zip(
        corpus_df["doc_key"],
        corpus_df["doc_id"]
    )
)

example_doc_ids = {}

for example_id, doc_keys in example_doc_keys.items():
    example_doc_ids[example_id] = [
        int(doc_key_to_id[key])
        for key in doc_keys
    ]

print("Corpus size:", len(corpus_df))
print("Document ID range:",
      int(corpus_df["doc_id"].min()),
      int(corpus_df["doc_id"].max()))

Corpus size: 62933
Document ID range: 0 62932


In [12]:
def get_supporting_doc_ids(record: Dict[str, Any]) -> List[int]:
    """
    Map HotpotQA supporting-fact titles to paragraph-level doc IDs.
    A supporting title corresponds to a context paragraph.
    """
    supporting_titles = set(record["supporting_titles"])

    result = []

    for context in record["contexts"]:
        title = context["title"]

        if title in supporting_titles:
            text = " ".join(context["sentences"]).strip()
            key = make_doc_key(title, text)

            if key not in doc_key_to_id:
                raise KeyError(
                    f"Supporting document missing from corpus: {title}"
                )

            result.append(int(doc_key_to_id[key]))

    return sorted(set(result))


def add_document_metadata(
    records: List[Dict[str, Any]]
) -> pd.DataFrame:

    rows = []

    for record in records:

        doc_ids = example_doc_ids[record["id"]]
        supporting_doc_ids = get_supporting_doc_ids(record)

        rows.append({
            "id": record["id"],
            "question": record["question"],
            "answer": record["answer"],
            "type": record["type"],
            "level": record["level"],
            "doc_ids": json.dumps(doc_ids),
            "supporting_doc_ids": json.dumps(supporting_doc_ids),
        })

    return pd.DataFrame(rows)


controller_train_meta = add_document_metadata(
    controller_train_records
)

controller_val_meta = add_document_metadata(
    controller_val_records
)

final_test_meta = add_document_metadata(
    final_test_records
)

print(controller_train_meta.head())

                         id  \
0  5ab3fd5255429976abd1bd15   
1  5a8f6bfe55429918e830d21c   
2  5a87943b5542993e715abfb8   
3  5a90a14755429916514e750a   
4  5ab561c3554299488d4d9972   

                                            question  \
0  Which city is larger, Pingxiang or Shijiazhuan...   
1  Jimmy Urine and Mark Hoppus, have which mutual...   
2  Sir Syed University of Engineering and Technol...   
3  Emory University and Syracuse University, are ...   
4                  What are Juniper and Heptacodium?   

                                              answer        type level  \
0  Shijiazhuang ( ; ), formerly romanized Shihkia...  comparison  easy   
1                   singer, songwriter, and musician  comparison  easy   
2                                                yes  comparison  easy   
3                                      United States  comparison  easy   
4                                             plants  comparison  easy   

                               

In [13]:
# Every selected example must have retrieval documents.
assert controller_train_meta["doc_ids"].map(
    lambda x: len(json.loads(x)) > 0
).all()

assert controller_val_meta["doc_ids"].map(
    lambda x: len(json.loads(x)) > 0
).all()

assert final_test_meta["doc_ids"].map(
    lambda x: len(json.loads(x)) > 0
).all()

# Every example must have at least one supporting document.
assert controller_train_meta["supporting_doc_ids"].map(
    lambda x: len(json.loads(x)) > 0
).all()

assert controller_val_meta["supporting_doc_ids"].map(
    lambda x: len(json.loads(x)) > 0
).all()

assert final_test_meta["supporting_doc_ids"].map(
    lambda x: len(json.loads(x)) > 0
).all()

# Ensure all referenced IDs exist.
valid_doc_ids = set(corpus_df["doc_id"].tolist())

for df in [
    controller_train_meta,
    controller_val_meta,
    final_test_meta,
]:

    for value in df["doc_ids"]:
        ids = json.loads(value)
        assert set(ids).issubset(valid_doc_ids)

    for value in df["supporting_doc_ids"]:
        ids = json.loads(value)
        assert set(ids).issubset(valid_doc_ids)

print("All corpus/example/supporting-document mappings are valid.")

All corpus/example/supporting-document mappings are valid.


In [14]:
corpus_to_save = corpus_df[
    [
        "doc_id",
        "doc_key",
        "title",
        "text",
        "token_count_words",
    ]
].copy()

corpus_to_save.to_parquet(
    ARTIFACT_DIR / "corpus.parquet",
    index=False
)

controller_train_meta.to_parquet(
    ARTIFACT_DIR / "controller_train.parquet",
    index=False
)

controller_val_meta.to_parquet(
    ARTIFACT_DIR / "controller_validation.parquet",
    index=False
)

final_test_meta.to_parquet(
    ARTIFACT_DIR / "final_test.parquet",
    index=False
)

corpus_mapping = corpus_df[
    [
        "doc_id",
        "doc_key",
        "title",
        "text",
        "source_example_ids",
    ]
].copy()

corpus_mapping["source_example_ids"] = (
    corpus_mapping["source_example_ids"]
    .apply(json.dumps)
)

corpus_mapping.to_parquet(
    ARTIFACT_DIR / "corpus_doc_mapping.parquet",
    index=False
)

print("Saved corpus and split metadata.")

Saved corpus and split metadata.


In [15]:
TOKEN_PATTERN = re.compile(r"[A-Za-z0-9]+")


def tokenize(text: str) -> List[str]:
    return TOKEN_PATTERN.findall(
        text.lower()
    )


print(tokenize(
    "Barack Obama was born in Honolulu, Hawaii."
))

['barack', 'obama', 'was', 'born', 'in', 'honolulu', 'hawaii']


In [16]:
print("Tokenizing corpus for BM25...")

tokenized_corpus = [
    tokenize(text)
    for text in tqdm(
        corpus_to_save["text"].tolist(),
        desc="BM25 tokenization"
    )
]

print("Building BM25 index...")

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index built.")

Tokenizing corpus for BM25...


BM25 tokenization:   0%|          | 0/62933 [00:00<?, ?it/s]

Building BM25 index...
BM25 index built.


In [17]:
bm25_path = ARTIFACT_DIR / "bm25.pkl"

with open(bm25_path, "wb") as f:
    pickle.dump(
        {
            "bm25": bm25,
            "tokenized_corpus": tokenized_corpus,
            "token_pattern": TOKEN_PATTERN.pattern,
            "bm25_k1": bm25.k1,
            "bm25_b": bm25.b,
        },
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("Saved:", bm25_path)
print("Size:",
      round(bm25_path.stat().st_size / (1024 ** 2), 2),
      "MB")

Saved: /kaggle/working/retrieval_artifacts/bm25.pkl
Size: 68.78 MB


In [18]:
print("Loading embedding model...")

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=device
)

print("Model loaded.")

embedding_dimension = embedding_model.get_sentence_embedding_dimension()

print("Embedding dimension:", embedding_dimension)

assert embedding_dimension == 384, (
    f"Unexpected BGE embedding dimension: {embedding_dimension}"
)

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded.
Embedding dimension: 384


/tmp/ipykernel_23/3822884391.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimension = embedding_model.get_sentence_embedding_dimension()


In [19]:
corpus_texts = (
    corpus_to_save["title"].astype(str)
    + " [SEP] "
    + corpus_to_save["text"].astype(str)
).tolist()

print("Documents to embed:", len(corpus_texts))

embedding_start = time.perf_counter()

document_embeddings = embedding_model.encode(
    corpus_texts,
    batch_size=EMBED_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
    device=device,
)

embedding_time = time.perf_counter() - embedding_start

document_embeddings = np.asarray(
    document_embeddings,
    dtype=np.float32
)

print("Embedding shape:", document_embeddings.shape)
print("Embedding dtype:", document_embeddings.dtype)
print(f"Embedding time: {embedding_time / 60:.2f} minutes")

Documents to embed: 62933


Batches:   0%|          | 0/492 [00:00<?, ?it/s]

Embedding shape: (62933, 384)
Embedding dtype: float32
Embedding time: 3.43 minutes


In [20]:
assert document_embeddings.ndim == 2
assert document_embeddings.shape[0] == len(corpus_to_save)
assert document_embeddings.shape[1] == embedding_dimension
assert document_embeddings.dtype == np.float32

norms = np.linalg.norm(
    document_embeddings[:100],
    axis=1
)

print("First 100 embedding norm range:",
      norms.min(),
      norms.max())

assert np.allclose(
    norms,
    1.0,
    atol=1e-3
), "Embeddings do not appear normalized."

print("Embedding integrity checks passed.")

First 100 embedding norm range: 0.99999994 1.0000001
Embedding integrity checks passed.


In [21]:
embedding_path = ARTIFACT_DIR / "dense_embeddings.npy"

np.save(
    embedding_path,
    document_embeddings
)

print("Saved:", embedding_path)
print("Size:",
      round(embedding_path.stat().st_size / (1024 ** 2), 2),
      "MB")

Saved: /kaggle/working/retrieval_artifacts/dense_embeddings.npy
Size: 92.19 MB


In [22]:
print("Building FAISS index...")

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(
    document_embeddings
)

print("FAISS index size:", faiss_index.ntotal)
print("Expected size:", len(corpus_to_save))

assert faiss_index.ntotal == len(corpus_to_save)

Building FAISS index...
FAISS index size: 62933
Expected size: 62933


In [23]:
faiss_path = ARTIFACT_DIR / "faiss.index"

faiss.write_index(
    faiss_index,
    str(faiss_path)
)

print("Saved:", faiss_path)
print("Size:",
      round(faiss_path.stat().st_size / (1024 ** 2), 2),
      "MB")

Saved: /kaggle/working/retrieval_artifacts/faiss.index
Size: 92.19 MB


In [24]:
def bm25_retrieve(
    query: str,
    bm25_index: BM25Okapi,
    corpus_df: pd.DataFrame,
    k: int = 5
) -> List[Dict[str, Any]]:
    """
    Retrieve top-k documents using BM25.
    """

    query_tokens = tokenize(query)

    scores = bm25_index.get_scores(
        query_tokens
    )

    k = min(k, len(scores))

    top_indices = np.argpartition(
        scores,
        -k
    )[-k:]

    top_indices = top_indices[
        np.argsort(scores[top_indices])[::-1]
    ]

    results = []

    for idx in top_indices:
        row = corpus_df.iloc[int(idx)]

        results.append({
            "doc_id": int(row["doc_id"]),
            "title": row["title"],
            "text": row["text"],
            "score": float(scores[idx]),
        })

    return results


def dense_retrieve(
    query: str,
    embedding_model: SentenceTransformer,
    index: faiss.Index,
    corpus_df: pd.DataFrame,
    k: int = 10,
) -> List[Dict[str, Any]]:
    """
    Retrieve top-k documents using dense BGE embeddings + FAISS.
    """

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
        device=device,
    ).astype(np.float32)

    scores, indices = index.search(
        query_embedding,
        min(k, index.ntotal)
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):
        if idx < 0:
            continue

        row = corpus_df.iloc[int(idx)]

        results.append({
            "doc_id": int(row["doc_id"]),
            "title": row["title"],
            "text": row["text"],
            "score": float(score),
        })

    return results

In [25]:
test_question = controller_val_records[0]["question"]

bm25_results = bm25_retrieve(
    test_question,
    bm25,
    corpus_to_save,
    k=BM25_K
)

print("Question:")
print(test_question)

print("\nBM25 results:")

for rank, result in enumerate(
    bm25_results,
    start=1
):
    print(
        f"\nRank {rank}"
        f"\nDoc ID: {result['doc_id']}"
        f"\nScore: {result['score']:.4f}"
        f"\nTitle: {result['title']}"
        f"\nText: {result['text'][:300]}"
    )

Question:
I Knew You Were Trouble and State of Grace are both written by what singer-songwriter?

BM25 results:

Rank 1
Doc ID: 10492
Score: 48.5750
Title: I Knew You Were Trouble
Text: "I Knew You Were Trouble" is a song recorded by American singer-songwriter Taylor Swift for her fourth studio album, "Red" (2012).  It was released on October 9, 2012, in the United States by Big Machine Records as the third promotional single from the album.  Later, "I Knew You Were Trouble" was re

Rank 2
Doc ID: 10486
Score: 42.6135
Title: State of Grace (Taylor Swift song)
Text: "State of Grace" is a song by American singer-songwriter Taylor Swift from her fourth studio album "Red" (2012).  It was released to the iTunes Store on October 16, 2012, in the United States by Big Machine Records as the fourth and final promotional single from the album.  It was the only promotion

Rank 3
Doc ID: 39334
Score: 32.6194
Title: I Never Knew You
Text: I Never Knew You is the second extended play by American rap

In [26]:
dense_results = dense_retrieve(
    test_question,
    embedding_model,
    faiss_index,
    corpus_to_save,
    k=DENSE_K
)

print("Question:")
print(test_question)

print("\nDense retrieval results:")

for rank, result in enumerate(
    dense_results,
    start=1
):
    print(
        f"\nRank {rank}"
        f"\nDoc ID: {result['doc_id']}"
        f"\nScore: {result['score']:.4f}"
        f"\nTitle: {result['title']}"
        f"\nText: {result['text'][:300]}"
    )

Question:
I Knew You Were Trouble and State of Grace are both written by what singer-songwriter?

Dense retrieval results:

Rank 1
Doc ID: 10486
Score: 0.7702
Title: State of Grace (Taylor Swift song)
Text: "State of Grace" is a song by American singer-songwriter Taylor Swift from her fourth studio album "Red" (2012).  It was released to the iTunes Store on October 16, 2012, in the United States by Big Machine Records as the fourth and final promotional single from the album.  It was the only promotion

Rank 2
Doc ID: 10492
Score: 0.7441
Title: I Knew You Were Trouble
Text: "I Knew You Were Trouble" is a song recorded by American singer-songwriter Taylor Swift for her fourth studio album, "Red" (2012).  It was released on October 9, 2012, in the United States by Big Machine Records as the third promotional single from the album.  Later, "I Knew You Were Trouble" was re

Rank 3
Doc ID: 46074
Score: 0.6438
Title: The Trouble with Love Is
Text: "The Trouble with Love Is" is a song by Amer

In [27]:
dense_results = dense_retrieve(
    test_question,
    embedding_model,
    faiss_index,
    corpus_to_save,
    k=DENSE_K
)

print("Question:")
print(test_question)

print("\nDense retrieval results:")

for rank, result in enumerate(
    dense_results,
    start=1
):
    print(
        f"\nRank {rank}"
        f"\nDoc ID: {result['doc_id']}"
        f"\nScore: {result['score']:.4f}"
        f"\nTitle: {result['title']}"
        f"\nText: {result['text'][:300]}"
    )

Question:
I Knew You Were Trouble and State of Grace are both written by what singer-songwriter?

Dense retrieval results:

Rank 1
Doc ID: 10486
Score: 0.7702
Title: State of Grace (Taylor Swift song)
Text: "State of Grace" is a song by American singer-songwriter Taylor Swift from her fourth studio album "Red" (2012).  It was released to the iTunes Store on October 16, 2012, in the United States by Big Machine Records as the fourth and final promotional single from the album.  It was the only promotion

Rank 2
Doc ID: 10492
Score: 0.7441
Title: I Knew You Were Trouble
Text: "I Knew You Were Trouble" is a song recorded by American singer-songwriter Taylor Swift for her fourth studio album, "Red" (2012).  It was released on October 9, 2012, in the United States by Big Machine Records as the third promotional single from the album.  Later, "I Knew You Were Trouble" was re

Rank 3
Doc ID: 46074
Score: 0.6438
Title: The Trouble with Love Is
Text: "The Trouble with Love Is" is a song by Amer

In [28]:
config = {
    "seed": SEED,
    "debug": DEBUG,

    "dataset": {
        "name": DATASET_NAME,
        "config": DATASET_CONFIG,
        "controller_train_size": CONTROLLER_TRAIN_SIZE,
        "controller_validation_size": CONTROLLER_VAL_SIZE,
        "final_test_size": FINAL_TEST_SIZE,
    },

    "models": {
        "embedding_model": EMBEDDING_MODEL_NAME,
    },

    "retrieval": {
        "bm25_k": BM25_K,
        "dense_k": DENSE_K,
        "embedding_dimension": int(embedding_dimension),
        "bm25_tokenizer_pattern": TOKEN_PATTERN.pattern,
    },

    "corpus": {
        "num_documents": int(len(corpus_to_save)),
        "deduplication": "SHA1(title + normalized paragraph text)",
        "granularity": "paragraph",
    },

    "runtime": {
        "python_version": platform.python_version(),
        "torch_version": torch.__version__,
        "device": device,
        "gpu": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),
    },
}

with open(
    ARTIFACT_DIR / "retrieval_config.json",
    "w"
) as f:
    json.dump(
        config,
        f,
        indent=2
    )

print(
    json.dumps(
        config,
        indent=2
    )
)

{
  "seed": 42,
  "debug": false,
  "dataset": {
    "name": "hotpotqa/hotpot_qa",
    "config": "distractor",
    "controller_train_size": 5000,
    "controller_validation_size": 1000,
    "final_test_size": 1000
  },
  "models": {
    "embedding_model": "BAAI/bge-small-en-v1.5"
  },
  "retrieval": {
    "bm25_k": 5,
    "dense_k": 10,
    "embedding_dimension": 384,
    "bm25_tokenizer_pattern": "[A-Za-z0-9]+"
  },
  "corpus": {
    "num_documents": 62933,
    "deduplication": "SHA1(title + normalized paragraph text)",
    "granularity": "paragraph"
  },
  "runtime": {
    "python_version": "3.12.13",
    "torch_version": "2.10.0+cu128",
    "device": "cuda",
    "gpu": "Tesla T4"
  }
}


In [29]:
corpus_stats = {
    "num_documents": int(len(corpus_to_save)),
    "avg_words_per_document": float(
        corpus_to_save["token_count_words"].mean()
    ),
    "median_words_per_document": float(
        corpus_to_save["token_count_words"].median()
    ),
    "max_words_per_document": int(
        corpus_to_save["token_count_words"].max()
    ),
    "min_words_per_document": int(
        corpus_to_save["token_count_words"].min()
    ),
    "num_controller_train_examples": int(
        len(controller_train_meta)
    ),
    "num_controller_validation_examples": int(
        len(controller_val_meta)
    ),
    "num_final_test_examples": int(
        len(final_test_meta)
    ),
}

with open(
    ARTIFACT_DIR / "corpus_stats.json",
    "w"
) as f:
    json.dump(
        corpus_stats,
        f,
        indent=2
    )

print(
    json.dumps(
        corpus_stats,
        indent=2
    )
)

{
  "num_documents": 62933,
  "avg_words_per_document": 87.04337946705226,
  "median_words_per_document": 78.0,
  "max_words_per_document": 1378,
  "min_words_per_document": 3,
  "num_controller_train_examples": 5000,
  "num_controller_validation_examples": 1000,
  "num_final_test_examples": 1000
}


In [30]:
artifact_readme = f"""
RETRIEVAL ARTIFACTS
===================

Project:
Cost-Aware Adaptive RAG

Created by:
Notebook 1 - 01_data_and_retrieval.ipynb

Dataset:
{DATASET_NAME}
Configuration:
{DATASET_CONFIG}

Embedding model:
{EMBEDDING_MODEL_NAME}

Corpus:
- Paragraph-level documents
- Deduplicated using SHA1(title + normalized paragraph text)
- Total documents: {len(corpus_to_save)}

Splits:
- Controller train: {len(controller_train_meta)}
- Controller validation: {len(controller_val_meta)}
- Final evaluation: {len(final_test_meta)}

Retrieval:
- BM25 top-k: {BM25_K}
- Dense retrieval top-k: {DENSE_K}
- Dense embedding dimension: {embedding_dimension}

Files:
- corpus.parquet
    Global paragraph retrieval corpus.

- bm25.pkl
    BM25 index and tokenizer state.

- dense_embeddings.npy
    Normalized BGE paragraph embeddings.

- faiss.index
    FAISS IndexFlatIP dense retrieval index.

- controller_train.parquet
    Controller-training questions with document mappings.

- controller_validation.parquet
    Controller-validation questions with document mappings.

- final_test.parquet
    Held-out final evaluation questions with document mappings.

- corpus_doc_mapping.parquet
    Additional paragraph metadata and source-example mapping.

- retrieval_config.json
    Exact configuration and runtime metadata.

- corpus_stats.json
    Corpus and split statistics.
"""

with open(
    ARTIFACT_DIR / "README.txt",
    "w"
) as f:
    f.write(artifact_readme.strip())

print(artifact_readme)


RETRIEVAL ARTIFACTS

Project:
Cost-Aware Adaptive RAG

Created by:
Notebook 1 - 01_data_and_retrieval.ipynb

Dataset:
hotpotqa/hotpot_qa
Configuration:
distractor

Embedding model:
BAAI/bge-small-en-v1.5

Corpus:
- Paragraph-level documents
- Deduplicated using SHA1(title + normalized paragraph text)
- Total documents: 62933

Splits:
- Controller train: 5000
- Controller validation: 1000
- Final evaluation: 1000

Retrieval:
- BM25 top-k: 5
- Dense retrieval top-k: 10
- Dense embedding dimension: 384

Files:
- corpus.parquet
    Global paragraph retrieval corpus.

- bm25.pkl
    BM25 index and tokenizer state.

- dense_embeddings.npy
    Normalized BGE paragraph embeddings.

- faiss.index
    FAISS IndexFlatIP dense retrieval index.

- controller_train.parquet
    Controller-training questions with document mappings.

- controller_validation.parquet
    Controller-validation questions with document mappings.

- final_test.parquet
    Held-out final evaluation questions with document ma

In [31]:
artifact_readme = f"""
RETRIEVAL ARTIFACTS
===================

Project:
Cost-Aware Adaptive RAG

Created by:
Notebook 1 - 01_data_and_retrieval.ipynb

Dataset:
{DATASET_NAME}
Configuration:
{DATASET_CONFIG}

Embedding model:
{EMBEDDING_MODEL_NAME}

Corpus:
- Paragraph-level documents
- Deduplicated using SHA1(title + normalized paragraph text)
- Total documents: {len(corpus_to_save)}

Splits:
- Controller train: {len(controller_train_meta)}
- Controller validation: {len(controller_val_meta)}
- Final evaluation: {len(final_test_meta)}

Retrieval:
- BM25 top-k: {BM25_K}
- Dense retrieval top-k: {DENSE_K}
- Dense embedding dimension: {embedding_dimension}

Files:
- corpus.parquet
    Global paragraph retrieval corpus.

- bm25.pkl
    BM25 index and tokenizer state.

- dense_embeddings.npy
    Normalized BGE paragraph embeddings.

- faiss.index
    FAISS IndexFlatIP dense retrieval index.

- controller_train.parquet
    Controller-training questions with document mappings.

- controller_validation.parquet
    Controller-validation questions with document mappings.

- final_test.parquet
    Held-out final evaluation questions with document mappings.

- corpus_doc_mapping.parquet
    Additional paragraph metadata and source-example mapping.

- retrieval_config.json
    Exact configuration and runtime metadata.

- corpus_stats.json
    Corpus and split statistics.
"""

with open(
    ARTIFACT_DIR / "README.txt",
    "w"
) as f:
    f.write(artifact_readme.strip())

print(artifact_readme)


RETRIEVAL ARTIFACTS

Project:
Cost-Aware Adaptive RAG

Created by:
Notebook 1 - 01_data_and_retrieval.ipynb

Dataset:
hotpotqa/hotpot_qa
Configuration:
distractor

Embedding model:
BAAI/bge-small-en-v1.5

Corpus:
- Paragraph-level documents
- Deduplicated using SHA1(title + normalized paragraph text)
- Total documents: 62933

Splits:
- Controller train: 5000
- Controller validation: 1000
- Final evaluation: 1000

Retrieval:
- BM25 top-k: 5
- Dense retrieval top-k: 10
- Dense embedding dimension: 384

Files:
- corpus.parquet
    Global paragraph retrieval corpus.

- bm25.pkl
    BM25 index and tokenizer state.

- dense_embeddings.npy
    Normalized BGE paragraph embeddings.

- faiss.index
    FAISS IndexFlatIP dense retrieval index.

- controller_train.parquet
    Controller-training questions with document mappings.

- controller_validation.parquet
    Controller-validation questions with document mappings.

- final_test.parquet
    Held-out final evaluation questions with document ma

In [32]:
for path in sorted(ARTIFACT_DIR.iterdir()):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)

        print(
            f"{path.name:35s}"
            f"{size_mb:10.2f} MB"
        )

README.txt                               0.00 MB
bm25.pkl                                68.78 MB
controller_train.parquet                 0.77 MB
controller_validation.parquet            0.16 MB
corpus.parquet                          22.95 MB
corpus_doc_mapping.parquet              23.13 MB
corpus_stats.json                        0.00 MB
dense_embeddings.npy                    92.19 MB
faiss.index                             92.19 MB
final_test.parquet                       0.15 MB
retrieval_config.json                    0.00 MB
